In [15]:
import os
path = r"C:\Users\prati\PycharmProjects\PythonProject\multimodel rag project\data\NovaCore_Multimodal_Company_Report_2026.pdf"
output_img_dir = "extracted_images"
os.makedirs(output_img_dir,exist_ok=True)

In [16]:
import hashlib
import os
import pymupdf
from langchain_core.documents import Document

def extract_pdf_content(path, output_img_dir="extracted_images"):
    os.makedirs(output_img_dir, exist_ok=True)
    doc = pymupdf.open(path)
    structured_documents = []
    seen_image_hashes = {}  

    for page_idx in range(len(doc)):
        page = doc[page_idx]
        page_number = page_idx + 1

        page_text = page.get_text("text")
        if page_text.strip():
            structured_documents.append(Document(
                page_content=page_text,
                metadata={
                    "page_number": page_number,
                    "source_type": "text"
                }
            ))

        tabs = page.find_tables()
        for tab in tabs:
            table_df = tab.extract()
            markdown_table = "\n".join([" | ".join(str(cell) for cell in row) for row in table_df])
            structured_documents.append(Document(
                page_content=f"### Table on Page {page_number}\n{markdown_table}",
                metadata={
                    "page_number": page_number,
                    "source_type": "table"
                }
            ))

        image_list = page.get_images(full=True)
        for image_index, image in enumerate(image_list):
            xref = image[0]
            if not page.get_image_rects(xref):
                continue
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]

            img_hash = hashlib.md5(image_bytes).hexdigest()

            if img_hash not in seen_image_hashes:
                image_filename = f"page_{page_number}_img_{image_index}.{image_ext}"
                image_filepath = os.path.join(output_img_dir, image_filename)

                with open(image_filepath, "wb") as f:
                    f.write(image_bytes)
                
                seen_image_hashes[img_hash] = image_filepath

    return structured_documents

In [ ]:
stuctured_docs = extract_pdf_content(path)
stuctured_docs

In [18]:
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os
import json
import base64

def generate_image_summaries(img_dir="extracted_images",cache_file="image_summaries_cache.json"):

    load_dotenv()

    vision_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)

    cache = {}
    if os.path.exists(cache_file):
        try :
            with open(cache_file,'r',encoding="utf-8") as f:
                cache = json.load(f)
        except json.JSONDecodeError:
            cache ={}

    image_summaries = []

    if not os.path.exists(img_dir):
        return image_summaries

    for filename in sorted(os.listdir(img_dir)):
        if filename.endswith((".png",".jpeg",".jpg")):
            image_path = os.path.join(img_dir,filename)

            if filename in cache:
                image_summaries.append(cache[filename])
                continue

            with open(image_path,"rb") as image_file:
                base64_img = base64.b64encode(image_file.read()).decode("utf-8")

            message = HumanMessage(
                content=[
                    {
                        "type":"text",
                        "text":"Transcribe this figure for a search index. Plain text only, max 250 words.\n1) Figure type and title.\n2) Every visible label, number and text, grouped logically (compact table for chart data).\n3) Mark values read from gridlines rather than printed with '~'.\nDo not add insights, calculations, colour/shape descriptions, or comments about cropping.",
                    },
                    {
                        "type":"image_url",
                        "image_url":{"url": f"data:image/jpeg;base64,{base64_img}"}
                    }
                ]
            )

            response = vision_llm.invoke([message])

            summary_data = {
                "filename":filename,
                "image_path":image_path,
                "summary":response.content
            }

            cache[filename] = summary_data
            image_summaries.append(summary_data)

    with open(cache_file,"w",encoding="utf-8") as f:
        json.dump(cache,f,indent=4,ensure_ascii=False)

    return image_summaries

In [19]:
image_summaries = generate_image_summaries(img_dir="extracted_images")

In [ ]:
image_summaries

In [21]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def build_final_multimodel_document(path,img_dir="extracted_images",cache_file="image_summaries_cache.json"):

    raw_extracted_docs = extract_pdf_content(path)

    image_summaries = generate_image_summaries(img_dir=img_dir,cache_file=cache_file)

    text_docs = [doc for doc in raw_extracted_docs if doc.metadata["source_type"] == "text"]
    table_docs = [doc for doc in raw_extracted_docs if doc.metadata["source_type"] == "table"]

    image_summary_docs = []
    for item in image_summaries:
        filename = item["filename"]
        page_num = int(filename.split("_")[1]) if "page" in filename else 1

        image_summary_docs.append(Document(
            page_content=item["summary"],
            metadata={
                "page_number":page_num,
                "source_type":"image_summary",
                "image_path":item["image_path"]
            }
        ))

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap=200,
        separators=["\n\n","\n"," ",""]
    )

    chunked_text_docs = text_splitter.split_documents(text_docs)

    final_document = chunked_text_docs + table_docs + image_summary_docs

    return final_document

In [22]:
final_document = build_final_multimodel_document(path,img_dir="extracted_images",cache_file="image_summaries_cache.json")

final_document

[Document(metadata={'page_number': 1, 'source_type': 'text'}, page_content='NovaCore Systems Ltd. - Fictional company report created for a multimodal RAG demonstration\nPage 1\nNOVACORE SYSTEMS LTD.\nFY2026 BUSINESS PERFORMANCE &\nOPERATIONS REPORT\nA realistic fictional company document designed for multimodal RAG testing\nHeadquarters\nSingapore\nEmployees\n1,480\n2026 Revenue\n$132.0M\nOperating Margin\n18.4%\nPrimary Sector\nIndustrial AI & IoT\nMarkets Served\n42 countries\nDemo note: All company names, metrics, people, and events in this report are fictional. The document intentionally mixes text, tables,\ncharts, and embedded images so it can be used to demonstrate multimodal retrieval and question answering.'),
 Document(metadata={'page_number': 2, 'source_type': 'text'}, page_content='NovaCore Systems Ltd. - Fictional company report created for a multimodal RAG demonstration\nPage 2\n1. Executive Overview\nNarrative text section - useful for text retrieval and grounded summari

In [23]:
import os
import hashlib
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

def index_documents_to_pinecone(final_document, index_name="multimodel-rag-project"):
    load_dotenv()

    PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
    
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    
    pc = Pinecone(api_key=PINECONE_API_KEY)

    index_name = "multimodel-rag-project"

    existing_index = [i["name"] for i in pc.list_indexes()]

    if index_name not in existing_index:
        pc.create_index(
            name=index_name,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )
        
    vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)
    
    doc_ids = []
    for doc in final_document:
        unique_string = f"{doc.metadata.get('source_type')}_{doc.metadata.get('page_number')}_{doc.page_content[:100]}"
        doc_id = hashlib.md5(unique_string.encode("utf-8")).hexdigest()
        doc_ids.append(doc_id)
        
    vectorstore.add_documents(documents=final_document, ids=doc_ids)
    
    print(f"Successfully upserted {len(final_document)} documents to Pinecone index '{index_name}'!")
    return vectorstore

vectorstore = index_documents_to_pinecone(final_document)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2016.47it/s]


Successfully upserted 26 documents to Pinecone index 'multimodel-rag-project'!


In [24]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

def hybrid_retriever(final_document,vectorstore):

    bm25retriever = BM25Retriever.from_documents(final_document)
    bm25retriever.k = 6

    vector_retriever = vectorstore.as_retriever(
        search_type = "similarity",
        search_kwargs = {"k":6}
    )

    hybrid_retriever = EnsembleRetriever(
     retrievers =[bm25retriever,vector_retriever],
     weights = [0.5,0.5]   
    )

    return hybrid_retriever

In [25]:
hybrid_retriever = hybrid_retriever(final_document,vectorstore)

In [26]:
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

def reranked_retriever(hybrid_retriever):

    model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

    compressor = CrossEncoderReranker(model=model,top_n=6)

    compression_retriever = ContextualCompressionRetriever(
        base_compressor = compressor,
        base_retriever = hybrid_retriever
    )

    return compression_retriever

final_retriever = reranked_retriever(hybrid_retriever)

Loading weights: 100%|██████████| 201/201 [00:01<00:00, 188.95it/s]


In [27]:
import base64

def encode_image(image_path):
    with open(image_path,"rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [ ]:
import os
import base64
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def multimodal_rag_answer(query, final_retriever):
    retrieved_docs = final_retriever.invoke(query)
    
    context_texts = []
    image_payloads = []
    
    for doc in retrieved_docs:
        source_type = doc.metadata.get("source_type")
        page_num = doc.metadata.get("page_number")
        
        if source_type in ["text", "table"]:
            context_texts.append(f"[Page {page_num} - {source_type.upper()}]:\n{doc.page_content}")
            
        elif source_type == "image_summary":
            img_path = doc.metadata.get("image_path")
            context_texts.append(f"[Page {page_num} - IMAGE SUMMARY]:\n{doc.page_content}")
            
            if img_path and os.path.exists(img_path):
                base64_image = encode_image(img_path)
                image_payloads.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{base64_image}"}
                })

    combined_context = "\n\n---\n\n".join(context_texts)
    
    prompt = f"""You are an expert enterprise AI assistant. Answer the user's question accurately using ONLY the provided context and embedded visual data.

Context:
{combined_context}

User Question: {query}
Answer:"""

    messages_content = [{"type": "text", "text": prompt}]
    
    for img in image_payloads:
        messages_content.append(img)

    response = client.chat.completions.create(
        model="qwen/qwen3.8-27b",  
        messages=[{"role": "user", "content": messages_content}],
        temperature=0.1,
        max_tokens=500
    )
    
    return response.choices[0].message.content

query = "how many windows penag facility building have?"
answer = multimodal_rag_answer(query, final_retriever)
print(answer)

In [ ]:
from pinecone import Pinecone

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY)
# Connect to your specific index and fetch stats
index = pc.Index("multimodel-rag-project")
stats = index.describe_index_stats()

# Print total vector count
print(f"Total vectors stored: {stats.get('total_vector_count', 0)}")

In [ ]:
import base64
import io
import json
import os
import re
import time
from typing import Any, Dict, List

import pandas as pd
from dotenv import load_dotenv
from groq import Groq
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from PIL import Image

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
VISION_MODEL_NAME = os.getenv("VISION_MODEL_NAME", "qwen/qwen3.8-27b")
QA_MODEL_NAME = os.getenv("QA_MODEL_NAME", "qwen/qwen3.8-27b")
JUDGE_MODEL_NAME = os.getenv("JUDGE_MODEL_NAME", "openai/gpt-oss-120b")
EXTRACTED_IMAGES_DIR = os.getenv("EXTRACTED_IMAGES_DIR", "extracted_images")

MAX_CHUNKS_TO_LLM = 5
MAX_CONTEXT_CHARS = 6000
MAX_IMAGES_TO_LLM = 1
MAX_IMAGE_DIM = 896
EVAL_SLEEP_SECONDS = 8


def strip_think_tags(text: str) -> str:
    if not text:
        return ""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def encode_image_resized_base64(image_path: str, max_dim: int = MAX_IMAGE_DIM) -> tuple[str, str]:
    with Image.open(image_path) as img:
        img = img.convert("RGBA")
        bg = Image.new("RGB", img.size, "white")
        bg.paste(img, mask=img.split()[-1])
        img = bg
        w, h = img.size
        if max(w, h) > max_dim:
            scale = max_dim / max(w, h)
            img = img.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.Resampling.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=85)
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        return b64, "image/jpeg"


def _extract_json_dict(text: str) -> dict:
    cleaned = strip_think_tags(text or "")
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", cleaned, flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                return {}
        return {}

In [ ]:
def _get_docs_for_answer(retriever_bundle: Any, query: str, max_chunks: int = MAX_CHUNKS_TO_LLM) -> List[Any]:
    if retriever_bundle is None:
        return []
    if isinstance(retriever_bundle, tuple):
        compressor, base_retriever = retriever_bundle
        initial_docs = base_retriever.invoke(query)
        docs = compressor.compress_documents(initial_docs, query)
    else:
        docs = retriever_bundle.invoke(query)
    return list(docs)[:max_chunks]


def _build_trimmed_context(docs: List[Any], max_chars: int = MAX_CONTEXT_CHARS) -> tuple[str, List[str], List[str]]:
    blocks: List[str] = []
    sources_set = set()
    image_names: List[str] = []
    used_chars = 0

    for idx, doc in enumerate(docs, start=1):
        meta = getattr(doc, "metadata", {}) or {}
        raw_page = meta.get("page_number", meta.get("page", 0))
        try:
            page_num = int(float(raw_page))
        except (ValueError, TypeError):
            page_num = 0

        ctype = meta.get("content_type") or meta.get("source_type", "text")
        sources_set.add(f"Page {page_num} ({ctype})")

        chunk_text = (getattr(doc, "page_content", "") or "").strip()
        block = f"[Context {idx} | Page {page_num} | {ctype}]\n{chunk_text}"

        if used_chars + len(block) > max_chars:
            remaining = max_chars - used_chars
            if remaining > 150:
                blocks.append(block[:remaining] + "\n...[truncated]")
            break

        blocks.append(block)
        used_chars += len(block) + 2

        img_path_meta = meta.get("image_path") or ""
        img_name = meta.get("image_filename") or (os.path.basename(img_path_meta.replace("\\", "/")) if img_path_meta else None)
        if img_name and img_name not in image_names:
            image_names.append(img_name)

    context_str = "\n\n".join(blocks) if blocks else "No relevant context found."
    return context_str, sorted(sources_set), image_names


def multimodal_rag_answer(
    query: str,
    retriever_bundle: Any,
    max_images: int = MAX_IMAGES_TO_LLM,
) -> Dict[str, Any]:
    t0 = time.time()
    docs = _get_docs_for_answer(retriever_bundle, query, max_chunks=MAX_CHUNKS_TO_LLM)
    context_text, sources, candidate_images = _build_trimmed_context(docs, max_chars=MAX_CONTEXT_CHARS)

    system_prompt = (
        "You are a precise corporate document assistant. Answer ONLY using the provided context "
        "and attached images. If the answer is not in the context or images, explicitly state that "
        "the information is not found in the report. Do not use outside knowledge or guess."
    )

    user_content: List[Dict[str, Any]] = [
        {
            "type": "text",
            "text": f"Context:\n{context_text}\n\nQuestion:\n{query}",
        }
    ]

    sent_images: List[str] = []
    for fname in candidate_images[:max_images]:
        img_path = os.path.join(EXTRACTED_IMAGES_DIR, fname)
        if os.path.exists(img_path):
            try:
                b64_data, mime = encode_image_resized_base64(img_path, max_dim=MAX_IMAGE_DIM)
                user_content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:{mime};base64,{b64_data}"},
                })
                sent_images.append(fname)
            except Exception:
                continue

    client = Groq(api_key=GROQ_API_KEY)
    response = client.chat.completions.create(
        model=QA_MODEL_NAME,
        temperature=0.1,
        max_tokens=512,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
    )

    raw_answer = response.choices[0].message.content or ""
    clean_answer = strip_think_tags(raw_answer)

    return {
        "query": query,
        "answer": clean_answer,
        "sources": sources,
        "images_sent": sent_images,
        "latency": round(time.time() - t0, 2),
        "context": context_text,
        "docs": docs
    }

In [ ]:
judge_llm = ChatGroq(
    model=JUDGE_MODEL_NAME,
    temperature=0.0,
    max_tokens=1500,
)


def _doc_page(doc) -> int | None:
    meta = getattr(doc, "metadata", {}) or {}
    val = meta.get("page_number", meta.get("page"))
    if val is None:
        return None
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return None


def compute_retrieval_metrics(docs: list, expected_pages: set[int], k: int | None = None) -> dict:
    if not expected_pages:
        return {"recall_at_k": None, "precision_at_k": None}

    top_docs = docs[:k] if k else docs
    if not top_docs:
        return {"recall_at_k": 0.0, "precision_at_k": 0.0}

    retrieved_pages = [_doc_page(d) for d in top_docs]
    valid_pages = {p for p in retrieved_pages if p is not None}

    recall = len(expected_pages & valid_pages) / len(expected_pages)
    relevant_chunks = sum(1 for p in retrieved_pages if p in expected_pages)
    precision = relevant_chunks / len(top_docs)

    return {
        "recall_at_k": round(recall, 3),
        "precision_at_k": round(precision, 3),
    }


def llm_as_a_judge(
    question: str,
    ground_truth: str,
    context_text: str,
    answer: str,
    should_abstain: bool = False,
) -> dict:
    if should_abstain:
        judge_prompt = f"""You are a strict RAG abstention judge.
The question below CANNOT be answered from the report.

Question: {question}
Generated Answer: {answer}

Did the system properly abstain (state that the information is not found/not mentioned in the report) WITHOUT guessing or fabricating numbers?
Return ONLY valid JSON:
{{
  "abstained_correctly": 1.0,
  "faithfulness": 1.0,
  "answer_relevance": 1.0,
  "critique": "short 1-sentence note"
}}
"""
    else:
        judge_prompt = f"""You are a strict RAG evaluation judge. Score each metric from 0.0 to 1.0 using ONLY the provided inputs.

Question: {question}
Ground Truth: {ground_truth}
Retrieved Context: {context_text}
Generated Answer: {answer}

Score these three metrics (floats in [0.0, 1.0]):
1. context_recall: How completely Retrieved Context contains information needed for Ground Truth.
2. faithfulness: How strictly every claim in Generated Answer is supported by Retrieved Context.
3. answer_relevance: How directly and accurately Generated Answer addresses the Question compared to Ground Truth.

Return ONLY valid JSON:
{{
  "context_recall": 0.0,
  "faithfulness": 0.0,
  "answer_relevance": 0.0,
  "critique": "short 1-sentence note"
}}
"""

    for _ in range(3):
        parsed = _extract_json_dict(judge_llm.invoke([HumanMessage(content=judge_prompt)]).content)
        if parsed:
            return parsed
    return {"critique": "JUDGE_PARSE_ERROR"}

In [35]:
GOLDEN_FILE = "multimodal_rag_golden_dataset.json"

with open(GOLDEN_FILE, "r", encoding="utf-8") as f:
    loaded_dataset = json.load(f)

print(f"Loaded {len(loaded_dataset)} items from {GOLDEN_FILE}")


def evaluate_golden_sample(sample: dict, retriever_bundle, answer_fn, k: int | None = None) -> dict:
    qid = sample.get("id", "")
    category = sample.get("category", "")
    question = sample["question"]
    ground_truth = sample["ground_truth"]
    expected_pages = set(sample.get("expected_pages") or [])
    should_abstain = bool(sample.get("should_abstain", False))

    t0 = time.time()
    result = answer_fn(question, retriever_bundle)
    retrieved_docs = result["docs"]
    context_text = result["context"]
    elapsed = round(time.time() - t0, 2)

    answer = result.get("answer", "")
    latency = result.get("latency", elapsed)
    ret_metrics = compute_retrieval_metrics(retrieved_docs, expected_pages, k=k)
    scores = llm_as_a_judge(
        question=question,
        ground_truth=ground_truth,
        context_text=context_text,
        answer=answer,
        should_abstain=should_abstain,
    )

    if should_abstain:
        abstained_ok = scores.get("abstained_correctly")
        passed = abstained_ok is not None and float(abstained_ok) >= 0.8
        return {
            "id": qid,
            "category": category,
            "should_abstain": True,
            "question": question,
            "recall_at_k": None,
            "precision_at_k": None,
            "context_recall": None,
            "faithfulness": scores.get("faithfulness"),
            "answer_relevance": scores.get("answer_relevance"),
            "abstention_acc": abstained_ok,
            "passed": passed,
            "latency_s": latency,
            "critique": scores.get("critique", ""),
            "answer": answer,
        }

    cr = scores.get("context_recall") or 0.0
    fa = scores.get("faithfulness") or 0.0
    ar = scores.get("answer_relevance") or 0.0
    passed = float(cr) >= 0.7 and float(fa) >= 0.8 and float(ar) >= 0.8
    must = sample.get("must_contain", [])
    passed = passed and all(m.lower() in answer.lower() for m in must)

    return {
        "id": qid,
        "category": category,
        "should_abstain": False,
        "question": question,
        "recall_at_k": ret_metrics["recall_at_k"],
        "precision_at_k": ret_metrics["precision_at_k"],
        "context_recall": cr,
        "faithfulness": fa,
        "answer_relevance": ar,
        "abstention_acc": None,
        "passed": passed,
        "latency_s": latency,
        "critique": scores.get("critique", ""),
        "answer": answer,
    }


def run_all_golden_tests(
    dataset: list[dict],
    retriever_bundle,
    answer_fn=multimodal_rag_answer,
    k: int | None = None,
    sleep_s: float = EVAL_SLEEP_SECONDS,
) -> pd.DataFrame:
    rows = []
    total = len(dataset)
    for idx, item in enumerate(dataset, start=1):
        print(f"[{idx}/{total}] Running {item['id']} ({item['category']})...")
        row = evaluate_golden_sample(item, retriever_bundle, answer_fn, k=k)
        rows.append(row)
        status = "PASS" if row["passed"] else "FAIL"
        print(f"   -> {status} | latency={row['latency_s']}s | {row['critique']}")
        if sleep_s > 0 and idx < total:
            time.sleep(sleep_s)

    df = pd.DataFrame(rows)
    metric_cols = [
        "recall_at_k",
        "precision_at_k",
        "context_recall",
        "faithfulness",
        "answer_relevance",
        "abstention_acc",
        "latency_s",
    ]
    means = df[metric_cols].mean(numeric_only=True).round(3)
    pass_rate = round(df["passed"].mean() * 100, 1)

    print("\n================ GOLDEN DATASET SUMMARY ================")
    print(f"  Overall Pass Rate : {df['passed'].sum()}/{total} ({pass_rate}%)")
    for col, val in means.items():
        print(f"  {col:18s}: {val}")
    print("========================================================")
    return df


results_df = run_all_golden_tests(loaded_dataset, final_retriever, multimodal_rag_answer, sleep_s=8.0)
results_df.to_csv(f"golden_eval_{int(time.time())}.csv", index=False)

results_df[
    ["id", "category", "passed", "recall_at_k", "precision_at_k", "context_recall", "faithfulness", "answer_relevance", "abstention_acc", "critique"]
]

Loaded 22 items from multimodal_rag_golden_dataset.json
[1/22] Running Q01 (metric_lookup)...


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3.8-27b` in organization `org_01kjc54qtnew7ba10beg3pwjm0` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199630, Requested 1158. Please try again in 5m40.416s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}